# ДЗ 7. Срезы orders.csv: groupby, pivot_table, merge
Срок: **до воскресенья 29.11, 24:00**.
Сдача: папка `hw07` репозитория `dsa-homework` (`hw07.ipynb`): `git add hw07` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать. Рядом с каждой долей или средним указывайте, на скольких заказах оно посчитано.

Инструменты: всё с уроков 1–7. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))
```

Ваши ответы и объяснения:

1. Ответ [4, 2] Потому что мы используем с как индекс, а сложение происходит по Х причем по одноименным элементам, далее оформляем все в список
2. Ответ [1, 1] [2, 1] Потому что мы с первом случае искользуем count он учитывае только заполненые ячейки, а во втором и пустые тоже, в итоге прибавляет все
3. Ответ 20, Потому что мы создаем таблицу по дву размерностям а в индексах и колоннах используются уникальные значения, а без указания aggfunc бертся среднее, (10+30)/2=20. Далее обращаемся к индексу А под столбцом 9
4. Ответ 2, 3,  Потому что в первом случае inner by default. В втором left. Left не дает пропасть строкам взамен может написать nan
5. Ответ 4, Тут во второй таблице мы видим 2 раза 1, это приводит к дублированию, то есть 3+1

In [9]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("orders.csv"), "| customers.csv:", os.path.exists("customers.csv"))

orders.csv: True | customers.csv: True


In [10]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [83]:
df["cancel"] = df["status"] == "отменён"
paid = df[df["status"] == "оплачен"]
cu = pd.read_csv("customers.csv")
len(paid), cu.shape

(4739, (2200, 4))

Можно вместо ячейки очистки взять свою функцию `clean()` из ДЗ 6. Проверка: 5 280 строк, 4 739 оплаченных.

## [База] 1. Сводка по городам одной таблицей
Через `agg` с именами: число заказов, доля отмен, средний срок доставки по городам. Добавьте столбец «выручка оплаченных»: `by_city["revenue"] = paid.groupby("city")["amount"].sum()` (Pandas выровняет по ярлыкам, урок 5). Отсортируйте по выручке.
Какой город выделяется сразу по двум показателям?

## Мы можем заметить как город Шымкент выделяется на фоне остальных, тем что средний срок доставки 4,45 дня что значительно больше остальных городов, также доля отмен там 20,6 процентов

In [14]:
# 1
by_city = df.groupby('city').agg(order=('order_id', 'count'), cancel_rate=('cancel', 'mean'), avg_delivery=('delivery_days', 'mean'))
by_city['revenue'] = paid.groupby('city')['amount'].sum()
by_city.sort_values('revenue', ascending=False)

,order,cancel_rate,avg_delivery,revenue
city,,,,
Алматы,2116,0.077505,1.997077,29144090.0
Астана,1339,0.088125,2.019246,17884750.0
Шымкент,776,0.206186,4.450928,8602660.0
Караганда,519,0.090559,1.941414,7020200.0
Актобе,387,0.105943,2.070496,5117380.0
Атырау,143,0.076923,2.036232,1893850.0


## [База] 2. ДЗ 4 одной строкой
В ДЗ 4 вы считали категории циклами и NumPy. Повторите через `pivot_table`: выручка оплаченных по категориям и месяцам, изменение в %, средний чек по категориям и месяцам.
Сверьте с ответами ДЗ 4: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре 33 600 ₸. Сколько строк кода было и сколько стало?

In [35]:
# 2
pt = paid.pivot_table(index='category', columns='month', values='amount', aggfunc=['sum', 'mean'], margins=True)
pt['diff_%'] = (pt['sum'][10] / pt['sum'][9] - 1) * 100
pt

sum                                  mean                \
month                 9          10         All             9            10   
category                                                                      
Дом           4997730.0   5125790.0  10123520.0  10499.432773  10768.466387   
Книги         1065300.0   1140910.0   2206210.0   4020.000000   3815.752508   
Красота       2610950.0   2678990.0   5289940.0   6187.085308   6158.597701   
Одежда        9167100.0   8505730.0  17672830.0  13520.796460  13045.598160   
Электроника  17402330.0  16968100.0  34370430.0  32772.749529  33600.198020   
All          35243410.0  34419520.0  69662930.0  14858.098651  14541.411069   

                             diff_%  
month                 All            
category                             
Дом          10633.949580  2.562363  
Книги         3911.719858  7.097531  
Красота       6172.625438  2.605948  
Одежда       13287.842105 -7.214604  
Электроника  33176.090734 -2.495241  
All          14699.921924 -2.337714

## [База] 3. Отмены и срок доставки
По `delivery_days`: число заказов и доля отмен. Во сколько раз отменяют чаще при доставке за 8 дней, чем за 1 день?
Что это значит для Шымкента, где в октябре доставка 5–8 дней?

## На восьмом дне доставки отменяют в 10.2 раза чаще чем в первый день

In [61]:
# 3
gt = df.groupby('delivery_days').agg(orders=('order_id', 'count'), cancel_rate=('cancel', 'mean'))
gt

,orders,cancel_rate
delivery_days,,
1.0,1557,0.042389
2.0,1570,0.079618
3.0,1590,0.133962
5.0,86,0.220930
6.0,120,0.216667
7.0,95,0.347368
8.0,104,0.432692


In [ ]:
print(f'На восьмом дне доставки отменяют в {(gt['cancel_rate'][8] / gt['cancel_rate'][1]):.1f} раза чаще чем в первый день')

На восьмом дне доставки отменяют в 10.2 раза чаще чем в первый день


 ## Как мы видим для только Шымкент имеет срок доставки больше ТРЕХ дней

In [71]:
gt = df.groupby(['city', 'delivery_days']).agg(orders=('order_id', 'count'), cancel_rate=('cancel', 'mean')).round(3)
gt

orders  cancel_rate
city      delivery_days                     
Актобе    1.0               116        0.052
          2.0               124        0.121
          3.0               143        0.140
Алматы    1.0               692        0.043
          2.0               675        0.073
          3.0               686        0.121
Астана    1.0               419        0.033
          2.0               436        0.083
          3.0               444        0.142
Атырау    1.0                42        0.048
          2.0                49        0.020
          3.0                47        0.149
Караганда 1.0               175        0.040
          2.0               174        0.098
          3.0               146        0.151
Шымкент   1.0               113        0.062
          2.0               112        0.062
          3.0               124        0.145
          5.0                86        0.221
          6.0               120        0.217
          7.0                95        0.347
          8.0               104        0.433

## [Стандарт] 4. Исправьте код от AI
AI посчитал изменение выручки по городам и пишет менеджеру: «В Шымкенте выручка выросла на 2,4%, проблем нет». В коде **две** ошибки. Найдите их, исправьте и напишите менеджеру правильную цифру.
```python
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))
```

In [80]:
# 4
p = paid.pivot_table(index="city", columns="month", values="amount", aggfunc='sum')
p["chg"] = (p[10] / p[9] - 1) * 100
print(p["chg"].round(3))

city
Актобе       -1.418
Алматы        0.979
Астана       -2.033
Атырау        4.964
Караганда    -7.588
Шымкент     -11.289
Name: chg, dtype: float64


## [Стандарт] 5. Возраст клиентов
Соедините заказы с клиентами (`merge`, `how="left"`, проверьте `len`). Возрастная группа: `m["age_group"] = m["age"] // 10 * 10`.
По группам: число заказов, доля отмен, средний и медианный чек оплаченных.
AI предлагает: «Группа 60 покупает дороже всех и чаще отменяет, сделаем для неё отдельную акцию». Какие возрасты попали в группу 60 и сколько в ней заказов? Подтверждает ли медиана вывод AI?

In [179]:
# 5
m = df.merge(cu, on='customer_id', how='left')
len(m), len(paid), m['customer_id'].isna().sum(), len(cu), len(df['customer_id'].unique())

(5280, 4739, np.int64(0), 2200, 2002)

In [187]:
m['age_group'] = m['age'] // 10 * 10
m_table = m.groupby('age_group').agg(orders=('order_id', 'count'), cancel_rate=('cancel', 'mean')).round(3)

m_paid = m[m['status'] == 'оплачен']

m_table['avg_order'] = m_paid.groupby('age_group')['amount'].mean()
m_table['median'] = m_paid.groupby('age_group')['amount'].median()
m_table

,orders,cancel_rate,avg_order,median
age_group,,,,
10,229,0.087,14044.736842,10700.0
20,1226,0.096,14130.415162,10100.0
30,1241,0.111,15142.339075,10670.0
40,1170,0.111,14792.971154,10050.0
50,1294,0.092,14671.795745,10240.0
60,120,0.133,16779.134615,10495.0


## [Стандарт] 6. Повторные покупки
Сколько заказов у каждого клиента (`groupby("customer_id").size()`)? Сколько клиентов сделали 1, 2, 3… заказа (`value_counts().sort_index()`)? Какая доля клиентов купила больше одного раза?
Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? Сколько клиентов из customers.csv не заказали ничего?

In [130]:
# 6
orders_count = df.groupby('customer_id').size()
orders_count

customer_id
C00001    4
C00002    1
C00003    2
C00004    2
C00005    4
         ..
C02196    1
C02197    3
C02198    1
C02199    2
C02200    1
Length: 2002, dtype: int64

In [131]:
orders_count.value_counts().sort_index()

1     501
2     563
3     434
4     293
5     127
6      58
7      15
8       9
9       1
10      1
Name: count, dtype: int64

In [133]:
more_than_one = (orders_count > 1).mean() * 100
print(f'Больше чем 1 заказ совершило {more_than_one:.2f}% клиентов')

Больше чем 1 заказ совершило 74.98% клиентов


In [139]:
active_month = df.groupby('customer_id')['month'].nunique()
print(f'{(active_month == 2).sum()} клиентов заказывали оба месяца')

1064 клиентов заказывали оба месяца


In [143]:
no_orders = ~cu['customer_id'].isin(df['customer_id'])
print(f'{no_orders.sum()} клиентов из справочника не заказывали ничего')

198 клиентов из справочника не заказывали ничего


## [Челлендж] 7. Сколько стоил сбой доставки в Шымкенте
Допустим, в октябре в Шымкенте было бы столько же заказов и тот же средний чек оплаченных, а доля отмен осталась как в сентябре. Какой была бы выручка Шымкента в октябре? Сколько компания потеряла из-за отмен?
Сравните потерю с падением всей компании (823 890 ₸). Что было бы с выручкой компании без сбоя?

In [230]:
# 7
sh = df[df['city'] == 'Шымкент']
sh_table = sh.groupby(['city', 'month']).agg(total=('amount', 'sum'), avg_order=('amount', 'mean'), cancel_rate=('cancel', 'mean'), order_count=('order_id', 'count')).unstack()
sh_table


total                avg_order               cancel_rate  \
month           9          10            9             10          9    
city                                                                    
Шымкент  4998660.0  5947300.0  13923.844011  14262.110312    0.089136   

                  order_count       
month          10          9    10  
city                                
Шымкент  0.306954         359  417

In [227]:
probably_oct = (sh_table['order_count'][9] * sh_table['avg_order'][9] * (1-sh_table['cancel_rate'][10])).round()
real_oct = (sh_table['order_count'][10] * sh_table['avg_order'][10] * (1-sh_table['cancel_rate'][10])).round()
print(probably_oct)

city
Шымкент    4558650.0
dtype: float64


In [235]:
paid_balance = paid.groupby(['city', 'month'])['amount'].sum()['Шымкент'].sum()
total_balance = df.groupby(['city', 'month'])['amount'].sum()['Шымкент'].sum()
loss = total_balance - paid_balance
print(loss)

2343300.0


## [Челлендж] 8. Органика −6,0%: сигнал или шум?
Разбейте клиентов из customers.csv **случайно** на три группы в тех же пропорциях, что и каналы (`np.random.default_rng(7)`, `rng.choice`, `map`). Посчитайте изменение выручки оплаченных окт/сент для каждой группы. Повторите 100 раз в цикле.
В каком диапазоне меняется выручка случайной группы размером с органику (5% и 95% перцентили)? Можно ли по «органика −6,0%» делать вывод о SEO?

In [ ]:
# 8